# Multivariate Example

This notebook runs a two-channel VAR example and compares the VI warm-start posterior with the NUTS posterior.

The model fitted here is the real multivariate pipeline: time-domain data are converted to Wishart frequency-domain statistics, a log-P-spline spectral matrix is fitted, and posterior PSD/coherence summaries are reconstructed from the sampled spline weights.

This is an advanced example. Start with the five-minute univariate guide before running it.

In [ ]:
! pip install -q --upgrade "LogPSplinePSD" "multimethod>=1.12,<2"

## Imports

In [ ]:
import os

os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")

import jax
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)

from log_psplines import PipelineConfig, fit
from log_psplines.arviz_utils import (
    get_multivar_posterior_psd_quantiles,
    get_multivar_vi_psd_quantiles,
)
from log_psplines.plotting import PSDMatrixPlotSpec, plot_psd_matrix
from log_psplines.example_datasets import VARMAData

## Simulate a two-channel VAR process

The coefficient matrices below match the VAR3 study used elsewhere in the repository, but this notebook uses a smaller sample size and shorter inference settings.

In [ ]:
varma = VARMAData(n_samples=2**14)
plt.plot(varma.data)

## Run VI warm start and NUTS

These settings are small enough for an interactive quickstart. Increase `N`, `n_warmup`, `n_samples`, `vi_steps`, and `n_knots` for a real analysis.

In [ ]:
config = PipelineConfig(
    n_knots=10,
    degree=2,
    diffMatrixOrder=2,
    n_warmup=500,
    n_samples=750,
    num_chains=2,
    method="nuts",
    vi_guide="diag",
    Nb=8,
    target_accept_prob=0.9,
    max_tree_depth=8,
    true_psd=varma.get_true_psd(),
    rng_key=7,
    verbose=True,
)

result = fit(varma.ts, config)
idata = result.idata

## Compare VI and NUTS posterior PSDs

In [ ]:
fig, axes = plot_psd_matrix(
    PSDMatrixPlotSpec(
        idata=idata,
        overlay_vi=True,
        label="NUTS 90% CI",
        vi_label="VI 90% CI",
        channel_labels=["x0", "x1"],
        show_knots=False,
        save=False,
        close=False,
    )
)
fig.suptitle("VAR3 posterior spectral matrix: NUTS with VI overlay", y=1.02)
plt.savefig("_static/var3_psd_matrix.png", bbox_inches="tight")
plt.show()

![](_static/var3_psd_matrix.png)

## What to check

- The diagonal PSD panels should be positive and broadly track the analytical PSD.
- VI and NUTS should have similar large-scale structure; exact agreement is not expected for this very short smoke-test run.
- For production runs, increase the warmup/draw counts and inspect the saved diagnostics from `PipelineConfig(outdir=...)`.